<a href="https://colab.research.google.com/github/514em/assignment2/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/My Drive/Colab Notebooks/assignment2')

['images',
 'assignment1.ipynb',
 'picture.jpg',
 '3D7BE144-0A71-4F0A-A4CD-C5EA6A22F0C7.jpg']

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/assignment2/'

# Section 1: Image Stitching [20 points]

In this section, we implement keypoint detection, feature matching, homography estimation, and image stitching using both pyramid blending and linear blending with feathering.

## Q1.1 - Load Images and Convert to Grayscale [3 points]

In [ ]:
# Q1.1: Load all 3 images, convert to grayscale, and display

# Load images from the Q1 folder
view1 = cv2.imread(path + 'Q1/view1.jpeg')
view2 = cv2.imread(path + 'Q1/view2.jpeg')
view3 = cv2.imread(path + 'Q1/view3.jpeg')

# Convert BGR (OpenCV default) to RGB for display
view1_rgb = cv2.cvtColor(view1, cv2.COLOR_BGR2RGB)
view2_rgb = cv2.cvtColor(view2, cv2.COLOR_BGR2RGB)
view3_rgb = cv2.cvtColor(view3, cv2.COLOR_BGR2RGB)

# Convert to grayscale
view1_gray = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
view2_gray = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)
view3_gray = cv2.cvtColor(view3, cv2.COLOR_BGR2GRAY)

# Display color and grayscale versions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].imshow(view1_rgb); axes[0, 0].set_title('View 1 (Color)'); axes[0, 0].axis('off')
axes[0, 1].imshow(view2_rgb); axes[0, 1].set_title('View 2 (Color)'); axes[0, 1].axis('off')
axes[0, 2].imshow(view3_rgb); axes[0, 2].set_title('View 3 (Color)'); axes[0, 2].axis('off')
axes[1, 0].imshow(view1_gray, cmap='gray'); axes[1, 0].set_title('View 1 (Grayscale)'); axes[1, 0].axis('off')
axes[1, 1].imshow(view2_gray, cmap='gray'); axes[1, 1].set_title('View 2 (Grayscale)'); axes[1, 1].axis('off')
axes[1, 2].imshow(view3_gray, cmap='gray'); axes[1, 2].set_title('View 3 (Grayscale)'); axes[1, 2].axis('off')
plt.suptitle('Q1.1 - Loaded Images', fontsize=14)
plt.tight_layout()
plt.show()
print(f'View 1 shape: {view1.shape}')
print(f'View 2 shape: {view2.shape}')
print(f'View 3 shape: {view3.shape}')

## Q1.2 - SIFT Keypoints and Descriptors [3 points]

In [ ]:
# Q1.2: Use SIFT to compute keypoints and descriptors for view1 and view2

# Create SIFT detector
sift = cv2.SIFT_create()

# Compute keypoints and descriptors simultaneously for both images
kp1, desc1 = sift.detectAndCompute(view1_gray, None)
kp2, desc2 = sift.detectAndCompute(view2_gray, None)

print(f'View 1: {len(kp1)} keypoints, descriptor shape: {desc1.shape}')
print(f'View 2: {len(kp2)} keypoints, descriptor shape: {desc2.shape}')

# (a) Visualize keypoints with orientations and scales using DRAW_RICH_KEYPOINTS
# cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS draws circles showing scale and orientation
view1_kp_img = cv2.drawKeypoints(view1_rgb.copy(), kp1, None,
                                   flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
view2_kp_img = cv2.drawKeypoints(view2_rgb.copy(), kp2, None,
                                   flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
ax1.imshow(view1_kp_img)
ax1.set_title(f'View 1 - SIFT Keypoints ({len(kp1)} keypoints)', fontsize=12)
ax1.axis('off')
ax2.imshow(view2_kp_img)
ax2.set_title(f'View 2 - SIFT Keypoints ({len(kp2)} keypoints)', fontsize=12)
ax2.axis('off')
plt.suptitle('Q1.2a - SIFT Keypoints with Orientations and Scales', fontsize=14)
plt.tight_layout()
plt.show()

# (b) Visualize descriptors: overlapping histograms of 10 descriptors from view1
plt.figure(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i in range(10):
    plt.plot(np.arange(128), desc1[i], alpha=0.7, color=colors[i], label=f'Descriptor {i+1}')
plt.xlabel('Dimension Index (0-127)')
plt.ylabel('Descriptor Value')
plt.title('Q1.2b - Overlapping Histograms of 10 SIFT Descriptors (View 1)')
plt.legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Q1.3 - Feature Matching with BFMatcher [2 points]

In [ ]:
# Q1.3: Match descriptors between view1 and view2 using BFMatcher

# Create BFMatcher with L2 norm (appropriate for SIFT) and cross-check for cleaner matches
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)

# Match descriptors: query=desc1 (view1), train=desc2 (view2)
matches = bf.match(desc1, desc2)

# Sort matches by distance (lower = better)
matches = sorted(matches, key=lambda x: x.distance)

print(f'Total matches found: {len(matches)}')
print(f'Top 10 match distances: {[round(m.distance, 2) for m in matches[:10]]}')

# Draw the top 10 best matches
img_matches = cv2.drawMatches(
    view1_rgb, kp1, view2_rgb, kp2, matches[:10], None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(20, 8))
plt.imshow(img_matches)
plt.title('Q1.3 - Top 10 Best Matches between View 1 and View 2 (BFMatcher)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Q1.4 - Homography Estimation with RANSAC [2 points]

In [ ]:
# Q1.4: Estimate homography with RANSAC and apply transformation to align view1 with view2

# Extract matching keypoint coordinates
# queryIdx = index in desc1 (view1), trainIdx = index in desc2 (view2)
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# Find homography H mapping view1 -> view2 using RANSAC
# ransacReprojThreshold=5.0 pixels: max reprojection error to classify as inlier
H, mask_ransac = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

print(f'Homography matrix H (view1 -> view2):')
print(H)
print(f'\nInliers: {mask_ransac.sum()} / {len(mask_ransac)} matches')

# Determine canvas size to contain both images after warping
h1, w1 = view1_rgb.shape[:2]
h2, w2 = view2_rgb.shape[:2]

# Transform view1 corners to view2's coordinate system
corners_v1 = np.float32([[0,0],[w1,0],[w1,h1],[0,h1]]).reshape(-1,1,2)
corners_v1_warped = cv2.perspectiveTransform(corners_v1, H)

# All corners (view1 warped + view2 in its own frame)
corners_v2 = np.float32([[0,0],[w2,0],[w2,h2],[0,h2]]).reshape(-1,1,2)
all_corners = np.concatenate([corners_v1_warped, corners_v2], axis=0)

[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

# Translation offsets so all coordinates are non-negative in the output canvas
offset_x = max(0, -x_min)
offset_y = max(0, -y_min)
canvas_w = x_max - x_min
canvas_h = y_max - y_min

print(f'\nCanvas size: {canvas_w} x {canvas_h}')
print(f'Offset: ({offset_x}, {offset_y})')

# Adjust H with the translation so view1 warps correctly into the canvas
T = np.array([[1, 0, offset_x], [0, 1, offset_y], [0, 0, 1]], dtype=np.float64)
H_adj = T @ H

# Warp view1 to aligned canvas
view1_warped = cv2.warpPerspective(view1_rgb, H_adj, (canvas_w, canvas_h))

# Display the warped view1
plt.figure(figsize=(14, 7))
plt.imshow(view1_warped)
plt.title('Q1.4 - Transformed View 1 (aligned to View 2 coordinate system)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Q1.5 - Pyramid Blending (Laplacian/DoG) to Stitch View 1 and View 2 [4 points]

In [ ]:
# Q1.5: Stitch the aligned view1 with view2 using Laplacian pyramid blending
# We implement the pyramid from scratch using only cv2.pyrDown() and cv2.pyrUp()

def build_gaussian_pyramid(img, levels):
    """Build a Gaussian pyramid by repeatedly downsampling."""
    pyramid = [img.astype(np.float32)]
    for _ in range(levels - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))
    return pyramid

def build_laplacian_pyramid(gauss_pyr):
    """
    Build a Laplacian (Difference-of-Gaussians) pyramid from a Gaussian pyramid.
    Each level L[i] = G[i] - pyrUp(G[i+1]), capturing band-pass detail at that scale.
    The top level is simply the coarsest Gaussian image.
    """
    laplacian = []
    for i in range(len(gauss_pyr) - 1):
        h, w = gauss_pyr[i].shape[:2]
        # Upsample the next level to match current level size exactly
        upsampled = cv2.pyrUp(gauss_pyr[i + 1], dstsize=(w, h))
        laplacian.append(gauss_pyr[i] - upsampled)
    laplacian.append(gauss_pyr[-1])  # top level = coarsest Gaussian
    return laplacian

def pyramid_stitch(imgA, imgB, stitch_col, levels=5):
    """
    Stitch two images using Laplacian pyramid blending.
    - Columns < stitch_col are taken from imgA (left image)
    - Columns >= stitch_col are taken from imgB (right image)
    By blending in the Laplacian domain at each scale, the seam is smoothed
    across all frequency bands, avoiding visible transitions.
    """
    imgA = imgA.astype(np.float32)
    imgB = imgB.astype(np.float32)

    # Step 1: Build Gaussian pyramids for both images
    gpA = build_gaussian_pyramid(imgA, levels)
    gpB = build_gaussian_pyramid(imgB, levels)

    # Step 2: Build Laplacian (DoG) pyramids for both images
    lpA = build_laplacian_pyramid(gpA)
    lpB = build_laplacian_pyramid(gpB)

    # Step 3: At each pyramid level, stitch lpA (left) and lpB (right) at scaled stitch_col
    blended_pyr = []
    for i in range(levels):
        h, w = lpA[i].shape[:2]
        # Scale the stitch column to match current pyramid level
        col = min(stitch_col // (2**i), w)
        level = np.zeros_like(lpA[i])
        level[:, :col] = lpA[i][:, :col]
        level[:, col:] = lpB[i][:, col:]
        blended_pyr.append(level)

    # Step 4: Reconstruct by collapsing the blended pyramid
    result = blended_pyr[-1].copy()
    for i in range(levels - 2, -1, -1):
        h, w = blended_pyr[i].shape[:2]
        result = cv2.pyrUp(result, dstsize=(w, h)) + blended_pyr[i]

    return np.clip(result, 0, 255).astype(np.uint8)


# Place view2 in the same canvas as view1_warped
view2_canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
view2_canvas[offset_y:offset_y+h2, offset_x:offset_x+w2] = view2_rgb

# Find the overlap region between the two images
v1_has_content = view1_warped.any(axis=2)
v2_has_content = view2_canvas.any(axis=2)
overlap_mask = v1_has_content & v2_has_content
overlap_cols = np.where(overlap_mask.any(axis=0))[0]

if len(overlap_cols) > 0:
    stitch_col = int(overlap_cols.mean())  # Use midpoint of overlap as seam
    print(f'Overlap region: columns {overlap_cols[0]} to {overlap_cols[-1]}')
    print(f'Stitch column (midpoint of overlap): {stitch_col}')
else:
    stitch_col = canvas_w // 2
    print('No overlap detected, using canvas midpoint as stitch column')

# Apply Laplacian pyramid blending
num_levels = 5
view12 = pyramid_stitch(view1_warped, view2_canvas, stitch_col, levels=num_levels)

# Display the stitched panorama
plt.figure(figsize=(18, 8))
plt.imshow(view12)
plt.title('Q1.5 - Stitched Panorama (view12) using Laplacian Pyramid Blending')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'view12 shape: {view12.shape}')

## Q1.6 - Linear Blending with Feathering to Stitch View12 and View3 [4 points]

In [ ]:
# Q1.6: Repeat SIFT + matching + homography for view12 and view3,
# then stitch using linear blending with feathering

# Convert view12 to grayscale for SIFT
view12_gray = cv2.cvtColor(view12, cv2.COLOR_RGB2GRAY)
view3_gray_q6 = cv2.cvtColor(view3, cv2.COLOR_BGR2GRAY)  # reuse from above
view3_rgb_q6  = cv2.cvtColor(view3, cv2.COLOR_BGR2RGB)

# Compute SIFT keypoints and descriptors for view12 and view3
kp12, desc12 = sift.detectAndCompute(view12_gray, None)
kp3, desc3  = sift.detectAndCompute(view3_gray_q6, None)

print(f'view12: {len(kp12)} keypoints | view3: {len(kp3)} keypoints')

# Match with BFMatcher
bf2 = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches23 = bf2.match(desc12, desc3)
matches23 = sorted(matches23, key=lambda x: x.distance)
print(f'Total matches (view12 vs view3): {len(matches23)}')

# Display top 10 matches
img_matches23 = cv2.drawMatches(
    view12, kp12, view3_rgb_q6, kp3, matches23[:10], None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
plt.figure(figsize=(20, 6))
plt.imshow(img_matches23)
plt.title('Top 10 Matches between view12 and view3')
plt.axis('off')
plt.tight_layout()
plt.show()

# Estimate homography: H23 maps keypoints from view12 -> view3
src_pts23 = np.float32([kp12[m.queryIdx].pt for m in matches23]).reshape(-1, 1, 2)
dst_pts23 = np.float32([kp3[m.trainIdx].pt  for m in matches23]).reshape(-1, 1, 2)
H23, mask23 = cv2.findHomography(src_pts23, dst_pts23, cv2.RANSAC, 5.0)
print(f'\nH23 inliers: {mask23.sum()} / {len(mask23)}')

# We want to warp view3 into view12's coordinate system: use H23_inv
H23_inv = np.linalg.inv(H23)

# Determine canvas size to hold both view12 and warped view3
h12, w12 = view12.shape[:2]
h3,  w3  = view3_rgb_q6.shape[:2]

corners_v3 = np.float32([[0,0],[w3,0],[w3,h3],[0,h3]]).reshape(-1,1,2)
corners_v3_in_v12 = cv2.perspectiveTransform(corners_v3, H23_inv)
corners_v12_frame = np.float32([[0,0],[w12,0],[w12,h12],[0,h12]]).reshape(-1,1,2)
all_corners23 = np.concatenate([corners_v3_in_v12, corners_v12_frame], axis=0)

[xmin23, ymin23] = np.int32(all_corners23.min(axis=0).ravel() - 0.5)
[xmax23, ymax23] = np.int32(all_corners23.max(axis=0).ravel() + 0.5)

off_x23 = max(0, -xmin23)
off_y23 = max(0, -ymin23)
cw23 = xmax23 - xmin23
ch23 = ymax23 - ymin23
print(f'Canvas for view123: {cw23} x {ch23}, offset: ({off_x23}, {off_y23})')

# Adjust H23_inv with translation offset
T23 = np.array([[1,0,off_x23],[0,1,off_y23],[0,0,1]], dtype=np.float64)
H23_inv_adj = T23 @ H23_inv

# Warp view3 into view12's coordinate frame
view3_warped = cv2.warpPerspective(view3_rgb_q6, H23_inv_adj, (cw23, ch23))

# Place view12 in the expanded canvas
view12_canvas = np.zeros((ch23, cw23, 3), dtype=np.uint8)
y12e = min(off_y23 + h12, ch23)
x12e = min(off_x23 + w12, cw23)
view12_canvas[off_y23:y12e, off_x23:x12e] = view12[:y12e-off_y23, :x12e-off_x23]

In [ ]:
# Linear blending with feathering for view12 + view3

def linear_blend_feathering(imgA, imgB):
    """
    Blend two same-sized canvas images using linear feathering in the overlap region.
    imgA: left/reference image canvas
    imgB: right/warped image canvas
    - Before overlap: use imgA exclusively
    - In overlap: linearly fade from imgA (alpha=1) to imgB (alpha=0)
    - After overlap: use imgB exclusively
    """
    h, w = imgA.shape[:2]

    # Identify where each image has non-zero content
    a_content = imgA.any(axis=2)
    b_content = imgB.any(axis=2)

    # Find column range of the overlap region
    overlap = a_content & b_content
    overlap_cols = np.where(overlap.any(axis=0))[0]

    if len(overlap_cols) == 0:
        # No overlap: simple composite
        result = imgA.astype(np.float32).copy()
        only_b = (~a_content) & b_content
        result[only_b] = imgB[only_b].astype(np.float32)
        return np.clip(result, 0, 255).astype(np.uint8)

    x_start = int(overlap_cols[0])
    x_end   = int(overlap_cols[-1])
    print(f'Overlap region: columns {x_start} to {x_end} ({x_end - x_start} px wide)')

    # Build a 1D alpha ramp: 1 at x_start -> 0 at x_end (imgA fades out)
    overlap_width = x_end - x_start + 1
    ramp = np.linspace(1.0, 0.0, overlap_width)  # shape: (overlap_width,)

    # Expand to (h, w, 1) for broadcasting
    alpha = np.ones((h, w), dtype=np.float32)
    alpha[:, x_start:x_end+1] = ramp   # transition zone
    alpha[:, x_end+1:] = 0.0            # right of overlap: full imgB
    alpha = alpha[:, :, np.newaxis]      # (h, w, 1)

    # Blend: result = alpha * imgA + (1 - alpha) * imgB
    imgA_f = imgA.astype(np.float32)
    imgB_f = imgB.astype(np.float32)
    result = alpha * imgA_f + (1.0 - alpha) * imgB_f

    return np.clip(result, 0, 255).astype(np.uint8)


# Apply linear blending with feathering
view123 = linear_blend_feathering(view12_canvas, view3_warped)

# Display the final panorama
plt.figure(figsize=(20, 8))
plt.imshow(view123)
plt.title('Q1.6 - Final Panorama (view123)\nView1+View2: Pyramid Blending | View12+View3: Linear Feathering')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Final panorama shape: {view123.shape}')

## Q1.7 - Analysis [2 points]

### Effectiveness of SIFT Keypoints

SIFT (Scale-Invariant Feature Transform) keypoints were highly effective at finding correspondences between the overlapping image pairs. SIFT is designed to be invariant to scale changes, rotation, and moderate illumination differences, making it well-suited for panorama stitching where the same 3D scene is captured from slightly different viewpoints. Each keypoint descriptor is a 128-dimensional histogram of gradient orientations, which provides a rich and distinctive signature.

In the top 10 matches displayed, most matches were accurate (corresponding to the same real-world points). However, a small number of mismatches (outliers) can appear, particularly in texturally repetitive regions (e.g., sky or uniform surfaces) where multiple keypoints have similar descriptors. The use of RANSAC in the homography estimation step helps filter out these outliers and compute a robust transformation from the inlier matches only.

### Pyramid Blending vs. Linear Blending

**Pyramid blending** (Laplacian pyramid / DoG-based) decomposes each image into multiple frequency bands using a Laplacian pyramid, performs the blending at each scale separately, and then reconstructs the result. The key advantage is that the blending width is adapted to each frequency: low-frequency (coarse) components are blended over a wide transition zone, while high-frequency (fine) details are blended over a narrow zone. This effectively eliminates both visible seams (caused by intensity/colour differences) and ghosting artefacts (caused by misalignment at fine scales). The result is a visually seamless blend.

**Linear blending with feathering** applies a simple linear gradient mask in the overlap region: the contribution of each image transitions smoothly from 100% to 0% across the overlap. This eliminates hard seams but can produce ghost artefacts near the seam when there is any misalignment between the two images, because high-frequency content from both images is simultaneously present in the blend zone. It is computationally simpler than pyramid blending and performs well when the images are well-aligned and the overlap is not too wide.

**Conclusion:** Pyramid blending generally produces better results because it handles both low-frequency colour/brightness differences and high-frequency texture/edge mismatches appropriately at each scale. Linear blending is a good approximation when images are very well-registered and the overlap region is narrow, but it tends to show ghosting when there is any residual parallax or misalignment.

# Section 2: Pug Detector [30 points]

In this section we implement a PCA-based Pug detector using eigenfaces. PCA is implemented from scratch using the snapshot method. All further processing uses **grayscale** images only.

## Q2.1 - Load All Q2 Images, Convert to Grayscale, and Display [2 points]

In [ ]:
# Q2.1: Load all images from the Q2 folder (train and val sets, pugs and loaves)
# Convert to grayscale; display both colour and grayscale versions

import glob

# ---- Discover images using glob ----
# Expected folder structure under path + 'Q2/':
#   train/pug_1.jpg ... pug_10.jpg   (training pugs)
#   train/loaf_1.jpg ...             (training loaves)
#   val/pug_*.jpg                    (validation pugs)
#   val/loaf_*.jpg                   (validation loaves)
train_pug_paths  = sorted(glob.glob(path + 'Q2/train/pug*.jpg'))
train_loaf_paths = sorted(glob.glob(path + 'Q2/train/loaf*.jpg'))
val_pug_paths    = sorted(glob.glob(path + 'Q2/val/pug*.jpg'))
val_loaf_paths   = sorted(glob.glob(path + 'Q2/val/loaf*.jpg'))

print(f'Training pugs  : {len(train_pug_paths)} images')
print(f'Training loaves: {len(train_loaf_paths)} images')
print(f'Val pugs       : {len(val_pug_paths)} images')
print(f'Val loaves     : {len(val_loaf_paths)} images')

def load_images(paths):
    """Load a list of image paths and return (colour_rgb_list, grayscale_list, name_list)."""
    rgbs, grays, names = [], [], []
    for p in paths:
        img_bgr = cv2.imread(p)
        rgbs.append(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        grays.append(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY))
        names.append(p.split('/')[-1])
    return rgbs, grays, names

train_pug_rgb,  train_pug_gray,  train_pug_names  = load_images(train_pug_paths)
train_loaf_rgb, train_loaf_gray, train_loaf_names = load_images(train_loaf_paths)
val_pug_rgb,    val_pug_gray,    val_pug_names    = load_images(val_pug_paths)
val_loaf_rgb,   val_loaf_gray,   val_loaf_names   = load_images(val_loaf_paths)

def display_group(rgb_list, gray_list, names, group_title):
    """
    Display a group of images in two rows: colour (top) and grayscale (bottom).
    Each column corresponds to one image.
    """
    n = len(rgb_list)
    if n == 0:
        print(f'No images in group: {group_title}')
        return
    fig, axes = plt.subplots(2, n, figsize=(max(3*n, 6), 6))
    # Handle single-image edge case
    if n == 1:
        axes = axes.reshape(2, 1)
    for i in range(n):
        axes[0, i].imshow(rgb_list[i])
        axes[0, i].set_title(names[i], fontsize=7)
        axes[0, i].axis('off')
        axes[1, i].imshow(gray_list[i], cmap='gray')
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Colour', fontsize=9)
    axes[1, 0].set_ylabel('Grayscale', fontsize=9)
    fig.suptitle(group_title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# Display all four groups
display_group(train_pug_rgb,  train_pug_gray,  train_pug_names,
              'Q2.1 — Training Pugs (colour top, grayscale bottom)')
display_group(train_loaf_rgb, train_loaf_gray, train_loaf_names,
              'Q2.1 — Training Loaves of Bread (colour top, grayscale bottom)')
display_group(val_pug_rgb,    val_pug_gray,    val_pug_names,
              'Q2.1 — Validation Pugs (colour top, grayscale bottom)')
display_group(val_loaf_rgb,   val_loaf_gray,   val_loaf_names,
              'Q2.1 — Validation Loaves of Bread (colour top, grayscale bottom)')

# ---- Resize all grayscale images to a common size for PCA ----
# We use the size of the first training pug image as the reference.
IMG_H, IMG_W = train_pug_gray[0].shape[:2]
print(f'\nReference image size: {IMG_W} x {IMG_H} pixels')

def resize_gray(gray_list, h, w):
    """Resize a list of grayscale images to (h, w) and return as a list."""
    return [cv2.resize(g, (w, h), interpolation=cv2.INTER_AREA) for g in gray_list]

train_pug_gray_r  = resize_gray(train_pug_gray,  IMG_H, IMG_W)
train_loaf_gray_r = resize_gray(train_loaf_gray, IMG_H, IMG_W)
val_pug_gray_r    = resize_gray(val_pug_gray,    IMG_H, IMG_W)
val_loaf_gray_r   = resize_gray(val_loaf_gray,   IMG_H, IMG_W)

print('All grayscale images resized to reference dimensions.')
print(f'Training set: {len(train_pug_gray_r)} pugs, {len(train_loaf_gray_r)} loaves')
print(f'Validation set: {len(val_pug_gray_r)} pugs, {len(val_loaf_gray_r)} loaves')

## Q2.2 - Mean Pug Face [5 points]

In [ ]:
# Q2.2: Compute and display the mean face of the pug training images (pug_1.jpg to pug_10.jpg)

# Step 1: Stack the 10 training pug grayscale images into a data matrix X
# Each image is first flattened to a 1D vector of length IMG_H * IMG_W
# X has shape (N, D) where N = number of training pugs, D = IMG_H * IMG_W
N_train = len(train_pug_gray_r)
D = IMG_H * IMG_W  # dimensionality of each flattened image

X = np.array([img.flatten().astype(np.float64) for img in train_pug_gray_r])
print(f'Data matrix X shape: {X.shape}  (N={N_train} images, D={D} pixels each)')

# Step 2: Compute the mean face = average pixel value across all N training pugs
# mean_face has shape (D,) — one average intensity per pixel position
mean_face = X.mean(axis=0)          # shape: (D,)
mean_face_img = mean_face.reshape(IMG_H, IMG_W)  # reshape to 2D for display

print(f'Mean face vector shape: {mean_face.shape}')
print(f'Pixel intensity range: [{mean_face.min():.1f}, {mean_face.max():.1f}]')

# Step 3: Display the mean pug face alongside the individual training images
fig, axes = plt.subplots(2, N_train // 2 + N_train % 2, figsize=(18, 8))
axes = axes.flatten()
for i, img in enumerate(train_pug_gray_r):
    axes[i].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[i].set_title(train_pug_names[i], fontsize=8)
    axes[i].axis('off')
# Hide any unused subplots
for j in range(N_train, len(axes)):
    axes[j].axis('off')
fig.suptitle('Q2.2 — Training Pug Images (Grayscale)', fontsize=13)
plt.tight_layout()
plt.show()

# Display the mean face
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Mean face
im0 = axes[0].imshow(mean_face_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Mean Pug Face', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Mean face with jet colourmap to highlight intensity variation
im1 = axes[1].imshow(mean_face_img, cmap='jet')
axes[1].set_title('Mean Pug Face (jet colourmap)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Pixel-intensity histogram of the mean face
axes[2].hist(mean_face.ravel(), bins=64, color='steelblue', edgecolor='none')
axes[2].set_xlabel('Pixel Intensity')
axes[2].set_ylabel('Count')
axes[2].set_title('Mean Face — Pixel Intensity Histogram', fontsize=12)

fig.suptitle('Q2.2 — Mean Pug Face (computed from 10 training images)', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Mean face computed from {N_train} training pugs.')
print(f'Mean pixel intensity : {mean_face.mean():.2f}')
print(f'Std of mean face     : {mean_face.std():.2f}')